# PPE Model Retraining Workbench

Use detection images and annotations stored in MinIO (`s3://models/ppe-detection/`) to fine-tune the YOLO PPE model.

**Env vars:** `AWS_S3_ENDPOINT`, `AWS_S3_BUCKET`, `PPE_MODEL_PREFIX`, `PPE_ENDPOINT`

In [ ]:
import os
from pathlib import Path

S3_ENDPOINT = os.environ.get("AWS_S3_ENDPOINT", "http://minio.industrial-edge-ml-workspace.svc:9000")
S3_BUCKET = os.environ.get("AWS_S3_BUCKET", "models")
MODEL_PREFIX = os.environ.get("PPE_MODEL_PREFIX", "ppe-detection/model")
PPE_ENDPOINT = os.environ.get("PPE_ENDPOINT", "http://yolo-ppe-serving:8080").rstrip("/")
DATA_PREFIX = os.environ.get("PPE_DATA_PREFIX", "ppe-detection/training-data")
LOCAL_DATA = Path("/opt/app-root/src/training-data")
LOCAL_DATA.mkdir(parents=True, exist_ok=True)

print("S3 endpoint:", S3_ENDPOINT)
print("Model prefix:", MODEL_PREFIX)
print("Data prefix:", DATA_PREFIX)
print("PPE endpoint:", PPE_ENDPOINT)

In [ ]:
import boto3

s3 = boto3.client(
    "s3",
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=os.environ.get("AWS_ACCESS_KEY_ID", "minio"),
    aws_secret_access_key=os.environ.get("AWS_SECRET_ACCESS_KEY", "minio123"),
    region_name=os.environ.get("AWS_DEFAULT_REGION", "us-east-1"),
)

resp = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=DATA_PREFIX, MaxKeys=20)
keys = [obj["Key"] for obj in resp.get("Contents", [])]
print(f"Found {len(keys)} objects under s3://{S3_BUCKET}/{DATA_PREFIX}")
for key in keys[:10]:
    print(" -", key)

## Next steps

1. Upload labeled images to `s3://models/ppe-detection/training-data/` during demos.
2. Fine-tune with Ultralytics YOLO (`yolo train data=...`).
3. Push updated `best.pt` to `s3://models/ppe-detection/model/` and restart InferenceService pods.